In [ ]:
#Importing necessary libraries:

import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import re
from pygam import LinearGAM, s, f, l
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import numpy as np
import scipy.stats as stats


In [ ]:
#Loading the data into dataframe

df_cimt = pd.read_stata("data/Hb1Ac_CIMT.dta")


print(df_cimt.shape)
list(df_cimt.columns)

In [ ]:
x = pd.to_numeric(df_cimt["meanmax_imt_mm_2_0"], errors="coerce").dropna()

print("cIMT N:", len(x))
print("cIMT mean:", x.mean())
print("cIMT SD:", x.std())

In [ ]:
#Basic check on CIMT variable

cimt_cols = [
    "mean_imt_mm_2_0",
    "max_imt_mm_2_0",
    "meanmax_imt_mm_2_0",
    "nlog_mean_imt_mm_2",
    "nlog_max_imt_mm_2",
    "nlog_meanmax_imt_mm_2"
]

#Summary stats
# df_cimt[cimt_cols].describe().T

#Missingnes
# df_cimt[cimt_cols].isna().sum().sort_values(ascending=False)

#Correlations
# df[cimt_cols].corr()

# #Skewness
# df[cimt_cols].skew().sort_values(ascending=False)

df_cimt[cimt_cols].hist(bins=50, figsize=(12, 8))

In [ ]:
#Loading in the PRS data
df_prs = pd.read_stata("data/UKB 71392 CMD Project.dta")


In [ ]:

#Checking for overlap between the two datsets

# overlap = df_cimt["n_eid"].isin(df_prs["n_eid"]).sum()
# overlap

#Checking for overlapping rows with CIMT data as well

primary_cimt = "nlog_meanmax_imt_mm_2"

cimt_available_df = df_cimt.dropna(subset=[primary_cimt]).copy()

overlap_cimt_available = cimt_available_df["n_eid"].isin(df_prs["n_eid"]).sum()

overlap_cimt_available

In [ ]:
#Merging the PRS data with the CIMT data
df = df_cimt.merge(
    df_prs,
    on="n_eid",
    how="inner"
)

# df.shape
list(df.columns)


In [ ]:
#Dropping all rows that have a missing PRS_ADHD value or cimt value
primary_cimt = "nlog_meanmax_imt_mm_2"
prs = "prs_adhd_std"

df_basic = df.dropna(subset=[prs, primary_cimt]).copy()

df_basic.shape

In [ ]:
#Cleaning the dataset 

#Fixing the sex column and turning it binary:
df_basic['sex'] = df_basic['sex_plink'].map({1: 0, 2: 1}) #0 => male and 1=> female

#Removing individuals with missing PRS and Key covariates:
main_vars = ['prs_adhd_std', 'n_21001_0_0', 'age_0_0', 'sex',
    'townsend', 'cursmoke_0', 'curr_alcohol_0',
    'PGC1', 'PGC2', 'PGC3', 'PGC4', 'PGC5', 'PGC6', 'PGC7', 'PGC8',
    'chip','nlog_meanmax_imt_mm_2']

df_model = df_basic.dropna(subset=main_vars).copy()

print(df_model.shape)
list(df_model.columns)

In [ ]:
#Scaling Age and SES factors

scale_cols = ['age_0_0', 'townsend']
scaler = StandardScaler()
df_model[scale_cols] = scaler.fit_transform(df_model[scale_cols])

In [ ]:
#Visualizing the CIMT distribution 

plt.figure(figsize=(10, 6))
sns.histplot(df_model['nlog_meanmax_imt_mm_2'], kde=True, bins=50, color='skyblue')
plt.title("Distribution of nlog CIMT")
plt.xlabel("nlog_meanmax_CIMT")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# exploring the CIMT distribution:
df_model['nlog_meanmax_imt_mm_2'].describe()
df_model['nlog_meanmax_imt_mm_2'].quantile([0.001, 0.005, 0.01, 0.05, 0.5, 0.95, 0.99, 0.995, 0.999])

In [ ]:
#Tertile analysis

#Plotting the WHR distribution into PRS tertiles to visualise trends:

df_model['prs_tertile'] = pd.qcut(df_model['prs_adhd_std'], q =3, labels = ["Low","medium","high"])

#Summary stats

CIMT_tertile_summary = (
    df_model
    .groupby('prs_tertile', observed=False)['nlog_meanmax_imt_mm_2']
    .agg(['count', 'mean', 'std', 'median', 'min', 'max'])
)
print(CIMT_tertile_summary)

# Boxplot of WHR by PRS tertile

plt.figure(figsize=(8, 5))
sns.boxplot(x='prs_tertile', y='nlog_meanmax_imt_mm_2', data=df_model, palette='Set2')
plt.title('CIMT across ADHD PRS Tertiles')
plt.xlabel('ADHD PRS Tertile')
plt.ylabel('CIMT')
plt.show()



In [ ]:
#Tertile analysis

#Plotting the WHR distribution into PRS tertiles to visualise trends:

df_model['prs_tertile'] = pd.qcut(df_model['prs_adhd_std'], q =3, labels = ["Low","medium","high"])

#Summary stats

CIMT_tertile_summary = (
    df_model
    .groupby('prs_tertile', observed=False)["meanmax_imt_mm_2_0"]
    .agg(['count', 'mean', 'std', 'median', 'min', 'max'])
)
print(CIMT_tertile_summary)

# Boxplot of WHR by PRS tertile

plt.figure(figsize=(8, 5))
sns.boxplot(x='prs_tertile', y="meanmax_imt_mm_2_0", data=df_model, palette='Set2')
plt.title('CIMT across ADHD PRS Tertiles')
plt.xlabel('ADHD PRS Tertile')
plt.ylabel('CIMT')
plt.show()

In [ ]:
# Model 1: CIMT ~ ADHD PRS + genetic covariates

# Filter and select relevant columns
cols = [
    'prs_adhd_std',
    'nlog_meanmax_imt_mm_2',
    'chip',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

df_model1 = df_basic[cols].dropna().copy()

# Define predictors (X) and outcome (y)
X = df_model1.drop(columns='nlog_meanmax_imt_mm_2')

# Convert chip into dummy variables
X = pd.get_dummies(X, columns=['chip'], drop_first=True, dtype=float)

# Ensure all predictors are numeric
X = X.apply(pd.to_numeric, errors='coerce')
X = X.dropna()

# Align outcome with cleaned predictors
y = df_model1.loc[X.index, 'nlog_meanmax_imt_mm_2']

# Add intercept and fit OLS model
X = sm.add_constant(X)
model1_cimt = sm.OLS(y, X).fit()

print(model1_cimt.summary())

# Predict and attach predictions back to dataframe
df_model1 = df_model1.loc[X.index].copy()
df_model1['cimt_predicted'] = model1_cimt.predict(X)

# Bin the PRS for smoother plotting
df_model1['prs_bin'] = pd.cut(df_model1['prs_adhd_std'], bins=100)

binned = (
    df_model1
    .groupby('prs_bin', observed=False)[['prs_adhd_std', 'cimt_predicted']]
    .mean()
    .dropna()
)

# Plot
plt.figure(figsize=(8, 6))

sns.scatterplot(
    x='prs_adhd_std',
    y='nlog_meanmax_imt_mm_2',
    data=df_model1,
    alpha=0.1,
    edgecolor=None,
    label='Raw Data'
)

# Smoothed OLS fit
sns.lineplot(
    x='prs_adhd_std',
    y='cimt_predicted',
    data=binned,
    color='red',
    label='OLS Fit (Smoothed)'
)

plt.title('Model 1: ADHD PRS + Genetic Covariates vs CIMT')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Log Meanmax CIMT')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Model 2: CIMT ~ ADHD PRS + age + sex + genetic covariates

# Filter and select relevant columns
cols = [
    'prs_adhd_std',
    'nlog_meanmax_imt_mm_2',
    'age_0_0',
    'sex',
    'chip',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

df_model2 = df_basic[cols].dropna().copy()

# Define predictors (X) and outcome (y)
X = df_model2.drop(columns='nlog_meanmax_imt_mm_2')

# Convert chip into dummy variables
X = pd.get_dummies(X, columns=['chip'], drop_first=True, dtype=float)

# Ensure all predictors are numeric
X = X.apply(pd.to_numeric, errors='coerce')
X = X.dropna()

# Align outcome with cleaned predictors
y = df_model2.loc[X.index, 'nlog_meanmax_imt_mm_2']

# Add intercept and fit OLS model
X = sm.add_constant(X)
model2_cimt = sm.OLS(y, X).fit()

print(model2_cimt.summary())

# Predict and attach predictions back to dataframe
df_model2 = df_model2.loc[X.index].copy()
df_model2['cimt_predicted'] = model2_cimt.predict(X)

# Bin the PRS for smoother plotting
df_model2['prs_bin'] = pd.cut(df_model2['prs_adhd_std'], bins=100)

binned = (
    df_model2
    .groupby('prs_bin', observed=False)[['prs_adhd_std', 'cimt_predicted']]
    .mean()
    .dropna()
)

# Plot
plt.figure(figsize=(8, 6))

sns.scatterplot(
    x='prs_adhd_std',
    y='nlog_meanmax_imt_mm_2',
    data=df_model2,
    alpha=0.1,
    edgecolor=None,
    label='Raw Data'
)

sns.lineplot(
    x='prs_adhd_std',
    y='cimt_predicted',
    data=binned,
    color='red',
    label='OLS Fit (Smoothed)'
)

plt.title('Model 2: ADHD PRS + Age, Sex and Genetic Covariates vs CIMT')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Log Meanmax CIMT')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Model 3: CIMT ~ ADHD PRS + age + sex + Townsend + smoking + alcohol + genetic covariates

cols = [
    'prs_adhd_std',
    'nlog_meanmax_imt_mm_2',
    'age_0_0',
    'sex',
    'townsend',
    'cursmoke_0',
    'curr_alcohol_0',
    'chip',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

df_model3 = df_basic[cols].dropna().copy()

# Scale continuous covariates
from sklearn.preprocessing import StandardScaler

scale_cols = ['age_0_0', 'townsend']
scaler = StandardScaler()
df_model3[scale_cols] = scaler.fit_transform(df_model3[scale_cols])

# Define predictors and outcome
X = df_model3.drop(columns='nlog_meanmax_imt_mm_2')

# Convert categorical variables into dummy variables
X = pd.get_dummies(
    X,
    columns=['chip', 'cursmoke_0', 'curr_alcohol_0'],
    drop_first=True,
    dtype=float
)

# Ensure all predictors are numeric
X = X.apply(pd.to_numeric, errors='coerce')
X = X.dropna()

# Align outcome with cleaned predictors
y = df_model3.loc[X.index, 'nlog_meanmax_imt_mm_2']

# Add intercept and fit OLS model
X = sm.add_constant(X)
model3_cimt = sm.OLS(y, X).fit()

print(model3_cimt.summary())

# Predict and attach predictions back to dataframe
df_model3 = df_model3.loc[X.index].copy()
df_model3['cimt_predicted'] = model3_cimt.predict(X)

# Bin PRS for smoother plotting
df_model3['prs_bin'] = pd.cut(df_model3['prs_adhd_std'], bins=100)

binned = (
    df_model3
    .groupby('prs_bin', observed=False)[['prs_adhd_std', 'cimt_predicted']]
    .mean()
    .dropna()
)

# Plot
plt.figure(figsize=(8, 6))

sns.scatterplot(
    x='prs_adhd_std',
    y='nlog_meanmax_imt_mm_2',
    data=df_model3,
    alpha=0.1,
    edgecolor=None,
    label='Raw Data'
)

sns.lineplot(
    x='prs_adhd_std',
    y='cimt_predicted',
    data=binned,
    color='red',
    label='OLS Fit (Smoothed)'
)

plt.title('Model 3: ADHD PRS + Demographic, Lifestyle and Genetic Covariates vs CIMT')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Log Meanmax CIMT')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Model 4: CIMT ~ ADHD PRS + age + sex + BMI + Townsend + smoking + alcohol + genetic covariates

cols = [
    'prs_adhd_std',
    'nlog_meanmax_imt_mm_2',
    'age_0_0',
    'sex',
    'n_21001_0_0',      # BMI
    'townsend',
    'cursmoke_0',
    'curr_alcohol_0',
    'chip',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

df_model4 = df_basic[cols].dropna().copy()

# Scale continuous covariates
from sklearn.preprocessing import StandardScaler

scale_cols = ['age_0_0', 'n_21001_0_0', 'townsend']
scaler = StandardScaler()
df_model4[scale_cols] = scaler.fit_transform(df_model4[scale_cols])

# Define predictors and outcome
X = df_model4.drop(columns='nlog_meanmax_imt_mm_2')

# Convert categorical variables into dummy variables
X = pd.get_dummies(
    X,
    columns=['chip', 'cursmoke_0', 'curr_alcohol_0'],
    drop_first=True,
    dtype=float
)

# Ensure all predictors are numeric
X = X.apply(pd.to_numeric, errors='coerce')
X = X.dropna()

# Align outcome with cleaned predictors
y = df_model4.loc[X.index, 'nlog_meanmax_imt_mm_2']

# Add intercept and fit OLS model
X = sm.add_constant(X)
model4_cimt = sm.OLS(y, X).fit()

print(model4_cimt.summary())

# Predict and attach predictions back to dataframe
df_model4 = df_model4.loc[X.index].copy()
df_model4['cimt_predicted'] = model4_cimt.predict(X)

# Bin PRS for smoother plotting
df_model4['prs_bin'] = pd.cut(df_model4['prs_adhd_std'], bins=100)

binned = (
    df_model4
    .groupby('prs_bin', observed=False)[['prs_adhd_std', 'cimt_predicted']]
    .mean()
    .dropna()
)

# Plot
plt.figure(figsize=(8, 6))

sns.scatterplot(
    x='prs_adhd_std',
    y='nlog_meanmax_imt_mm_2',
    data=df_model4,
    alpha=0.1,
    edgecolor=None,
    label='Raw Data'
)

sns.lineplot(
    x='prs_adhd_std',
    y='cimt_predicted',
    data=binned,
    color='red',
    label='OLS Fit (Smoothed)'
)

plt.title('Model 4: ADHD PRS + BMI and Full Covariate Adjustment vs CIMT')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Log Meanmax CIMT')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Model 4: CIMT ~ ADHD PRS + age + sex + WHR + Townsend + smoking + alcohol + genetic covariates

cols = [
    'prs_adhd_std',
    'nlog_meanmax_imt_mm_2',
    'age_0_0',
    'sex',
    'whr_0',      # BMI
    'townsend',
    'cursmoke_0',
    'curr_alcohol_0',
    'chip',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

df_model5 = df_basic[cols].dropna().copy()


# Define predictors and outcome
X = df_model5.drop(columns='nlog_meanmax_imt_mm_2')

# Convert categorical variables into dummy variables
X = pd.get_dummies(
    X,
    columns=['chip', 'cursmoke_0', 'curr_alcohol_0'],
    drop_first=True,
    dtype=float
)

# Ensure all predictors are numeric
X = X.apply(pd.to_numeric, errors='coerce')
X = X.dropna()

# Align outcome with cleaned predictors
y = df_model5.loc[X.index, 'nlog_meanmax_imt_mm_2']

# Add intercept and fit OLS model
X = sm.add_constant(X)
model5_cimt = sm.OLS(y, X).fit()

print(model5_cimt.summary())

# Predict and attach predictions back to dataframe
df_model5 = df_model5.loc[X.index].copy()
df_model5['cimt_predicted'] = model5_cimt.predict(X)

# Bin PRS for smoother plotting
df_model5['prs_bin'] = pd.cut(df_model5['prs_adhd_std'], bins=100)

binned = (
    df_model5
    .groupby('prs_bin', observed=False)[['prs_adhd_std', 'cimt_predicted']]
    .mean()
    .dropna()
)

# Plot
plt.figure(figsize=(8, 6))

sns.scatterplot(
    x='prs_adhd_std',
    y='nlog_meanmax_imt_mm_2',
    data=df_model5,
    alpha=0.1,
    edgecolor=None,
    label='Raw Data'
)

sns.lineplot(
    x='prs_adhd_std',
    y='cimt_predicted',
    data=binned,
    color='red',
    label='OLS Fit (Smoothed)'
)

plt.title('Model 5: ADHD PRS + WHR and Full Covariate Adjustment vs CIMT')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Log Meanmax CIMT')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# GAM 1: CIMT ~ ADHD PRS + genetic covariates + chip

from pygam import LinearGAM, s, f
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Select columns
cols = [
    'prs_adhd_std',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8',
    'chip',
    'nlog_meanmax_imt_mm_2'
]

df_gam1 = df_basic[cols].copy()

# Replace inf values with NaN
df_gam1 = df_gam1.replace([np.inf, -np.inf], np.nan)

# Drop missing values from both predictors and outcome
df_gam1 = df_gam1.dropna().copy()

# Convert chip to categorical numeric codes if needed
df_gam1['chip'] = df_gam1['chip'].astype('category').cat.codes

# Prepare X and y
X_gam1 = df_gam1[
    [
        'prs_adhd_std',
        'PGC1', 'PGC2', 'PGC3', 'PGC4',
        'PGC5', 'PGC6', 'PGC7', 'PGC8',
        'chip'
    ]
].values

y_gam1 = df_gam1['nlog_meanmax_imt_mm_2'].values

# Check that there are no NaN or inf values left
print("NaN in X:", np.isnan(X_gam1).sum())
print("Inf in X:", np.isinf(X_gam1).sum())
print("NaN in y:", np.isnan(y_gam1).sum())
print("Inf in y:", np.isinf(y_gam1).sum())
print("Shape:", X_gam1.shape, y_gam1.shape)

# Fit GAM
gam1_cimt = LinearGAM(
    s(0) + s(1) + s(2) + s(3) + s(4) +
    s(5) + s(6) + s(7) + s(8) +
    f(9)
).fit(X_gam1, y_gam1)

# Generate PRS grid for term 0
XX = gam1_cimt.generate_X_grid(term=0)

# Predict adjusted CIMT and confidence intervals
pdep = gam1_cimt.predict(XX)
conf_int = gam1_cimt.confidence_intervals(XX)

# Plot
plt.figure(figsize=(10, 6))

plt.plot(
    XX[:, 0],
    pdep,
    color='red',
    label='GAM fit'
)

plt.fill_between(
    XX[:, 0],
    conf_int[:, 0],
    conf_int[:, 1],
    alpha=0.3,
    color='blue',
    label='95% CI'
)

plt.title("GAM 1: ADHD PRS and CIMT adjusted for genetic covariates")
plt.xlabel("Standardised ADHD PRS")
plt.ylabel("Predicted log meanmax CIMT")
plt.legend()
plt.tight_layout()
plt.show()

# Print summary
gam1_cimt.summary()

In [ ]:
# GAM 2: CIMT ~ ADHD PRS + age + sex + genetic covariates + chip
# Partial dependence plot for ADHD PRS

from pygam import LinearGAM, s, f
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Select columns
cols = [
    'prs_adhd_std',
    'age_0_0',
    'sex',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8',
    'chip',
    'nlog_meanmax_imt_mm_2'
]

df_gam2 = df_basic[cols].copy()

# Replace inf values with NaN
df_gam2 = df_gam2.replace([np.inf, -np.inf], np.nan)

# Drop missing values
df_gam2 = df_gam2.dropna().copy()

# Convert categorical variables to numeric codes
df_gam2['sex'] = df_gam2['sex'].astype('category').cat.codes
df_gam2['chip'] = df_gam2['chip'].astype('category').cat.codes

# Prepare X and y
X_gam2 = df_gam2[
    [
        'prs_adhd_std',
        'age_0_0',
        'sex',
        'PGC1', 'PGC2', 'PGC3', 'PGC4',
        'PGC5', 'PGC6', 'PGC7', 'PGC8',
        'chip'
    ]
].values

y_gam2 = df_gam2['nlog_meanmax_imt_mm_2'].values

# Check missingness and shape
print("NaN in X:", np.isnan(X_gam2).sum())
print("Inf in X:", np.isinf(X_gam2).sum())
print("NaN in y:", np.isnan(y_gam2).sum())
print("Inf in y:", np.isinf(y_gam2).sum())
print("Shape:", X_gam2.shape, y_gam2.shape)

# Fit GAM
# Terms:
# s(0) = ADHD PRS
# s(1) = age
# f(2) = sex
# s(3) to s(10) = genetic PCs
# f(11) = chip

gam2_cimt = LinearGAM(
    s(0) +
    s(1) +
    f(2) +
    s(3) + s(4) + s(5) + s(6) +
    s(7) + s(8) + s(9) + s(10) +
    f(11)
).fit(X_gam2, y_gam2)

# Print summary
gam2_cimt.summary()

# Partial dependence plot for ADHD PRS
term = 0

XX = gam2_cimt.generate_X_grid(term=term)

pdep, confi = gam2_cimt.partial_dependence(
    term=term,
    X=XX,
    width=0.95
)

plt.figure(figsize=(10, 6))

plt.plot(
    XX[:, term],
    pdep,
    color='red',
    label='ADHD PRS smooth'
)

plt.fill_between(
    XX[:, term],
    confi[:, 0],
    confi[:, 1],
    alpha=0.3,
    color='blue',
    label='95% CI'
)

plt.axhline(
    0,
    color='black',
    linestyle='--',
    linewidth=1
)

plt.title("GAM 2: Partial Effect of ADHD PRS on CIMT")
plt.xlabel("Standardised ADHD PRS")
plt.ylabel("Partial effect on log meanmax CIMT")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# GAM 3: CIMT ~ ADHD PRS + age + sex + Townsend + smoking + alcohol + genetic covariates + chip
# Partial dependence plot for ADHD PRS

from pygam import LinearGAM, s, f
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Select columns
cols = [
    'prs_adhd_std',
    'age_0_0',
    'sex',
    'townsend',
    'cursmoke_0',
    'curr_alcohol_0',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8',
    'chip',
    'nlog_meanmax_imt_mm_2'
]

df_gam3 = df_basic[cols].copy()

# Replace inf values with NaN
df_gam3 = df_gam3.replace([np.inf, -np.inf], np.nan)

# Drop missing values
df_gam3 = df_gam3.dropna().copy()

# Convert categorical variables to numeric codes
df_gam3['sex'] = df_gam3['sex'].astype('category').cat.codes
df_gam3['cursmoke_0'] = df_gam3['cursmoke_0'].astype('category').cat.codes
df_gam3['curr_alcohol_0'] = df_gam3['curr_alcohol_0'].astype('category').cat.codes
df_gam3['chip'] = df_gam3['chip'].astype('category').cat.codes

# Prepare X and y
X_gam3 = df_gam3[
    [
        'prs_adhd_std',
        'age_0_0',
        'sex',
        'townsend',
        'cursmoke_0',
        'curr_alcohol_0',
        'PGC1', 'PGC2', 'PGC3', 'PGC4',
        'PGC5', 'PGC6', 'PGC7', 'PGC8',
        'chip'
    ]
].values

y_gam3 = df_gam3['nlog_meanmax_imt_mm_2'].values

# Check missingness and shape
print("NaN in X:", np.isnan(X_gam3).sum())
print("Inf in X:", np.isinf(X_gam3).sum())
print("NaN in y:", np.isnan(y_gam3).sum())
print("Inf in y:", np.isinf(y_gam3).sum())
print("Shape:", X_gam3.shape, y_gam3.shape)

# Fit GAM
# Terms:
# s(0) = ADHD PRS
# s(1) = age
# f(2) = sex
# s(3) = Townsend
# f(4) = current smoking
# f(5) = current alcohol
# s(6) to s(13) = genetic PCs
# f(14) = chip

gam3_cimt = LinearGAM(
    s(0) +
    s(1) +
    f(2) +
    s(3) +
    f(4) +
    f(5) +
    s(6) + s(7) + s(8) + s(9) +
    s(10) + s(11) + s(12) + s(13) +
    f(14)
).fit(X_gam3, y_gam3)

# Print summary
gam3_cimt.summary()

# Partial dependence plot for ADHD PRS
term = 0

XX = gam3_cimt.generate_X_grid(term=term)

pdep, confi = gam3_cimt.partial_dependence(
    term=term,
    X=XX,
    width=0.95
)

plt.figure(figsize=(10, 6))

plt.plot(
    XX[:, term],
    pdep,
    color='red',
    label='ADHD PRS smooth'
)

plt.fill_between(
    XX[:, term],
    confi[:, 0],
    confi[:, 1],
    alpha=0.3,
    color='blue',
    label='95% CI'
)

plt.axhline(
    0,
    color='black',
    linestyle='--',
    linewidth=1
)

plt.title("GAM 3: Partial Effect of ADHD PRS on CIMT")
plt.xlabel("Standardised ADHD PRS")
plt.ylabel("Partial effect on log meanmax CIMT")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# GAM 4: CIMT ~ ADHD PRS + age + sex + BMI + Townsend + smoking + alcohol + genetic covariates + chip
# Partial dependence plot for ADHD PRS

from pygam import LinearGAM, s, f
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Select columns
cols = [
    'prs_adhd_std',
    'age_0_0',
    'sex',
    'n_21001_0_0',      # BMI
    'townsend',
    'cursmoke_0',
    'curr_alcohol_0',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8',
    'chip',
    'nlog_meanmax_imt_mm_2'
]

df_gam4 = df_basic[cols].copy()

# Replace inf values with NaN
df_gam4 = df_gam4.replace([np.inf, -np.inf], np.nan)

# Drop missing values
df_gam4 = df_gam4.dropna().copy()

# Convert categorical variables to numeric codes
df_gam4['sex'] = df_gam4['sex'].astype('category').cat.codes
df_gam4['cursmoke_0'] = df_gam4['cursmoke_0'].astype('category').cat.codes
df_gam4['curr_alcohol_0'] = df_gam4['curr_alcohol_0'].astype('category').cat.codes
df_gam4['chip'] = df_gam4['chip'].astype('category').cat.codes

# Prepare X and y
X_gam4 = df_gam4[
    [
        'prs_adhd_std',
        'age_0_0',
        'sex',
        'n_21001_0_0',
        'townsend',
        'cursmoke_0',
        'curr_alcohol_0',
        'PGC1', 'PGC2', 'PGC3', 'PGC4',
        'PGC5', 'PGC6', 'PGC7', 'PGC8',
        'chip'
    ]
].values

y_gam4 = df_gam4['nlog_meanmax_imt_mm_2'].values

# Check missingness and shape
print("NaN in X:", np.isnan(X_gam4).sum())
print("Inf in X:", np.isinf(X_gam4).sum())
print("NaN in y:", np.isnan(y_gam4).sum())
print("Inf in y:", np.isinf(y_gam4).sum())
print("Shape:", X_gam4.shape, y_gam4.shape)

# Fit GAM
# Terms:
# s(0) = ADHD PRS
# s(1) = age
# f(2) = sex
# s(3) = BMI
# s(4) = Townsend
# f(5) = current smoking
# f(6) = current alcohol
# s(7) to s(14) = genetic PCs
# f(15) = chip

gam4_cimt = LinearGAM(
    s(0) +
    s(1) +
    f(2) +
    s(3) +
    s(4) +
    f(5) +
    f(6) +
    s(7) + s(8) + s(9) + s(10) +
    s(11) + s(12) + s(13) + s(14) +
    f(15)
).fit(X_gam4, y_gam4)

# Print summary
gam4_cimt.summary()

# Partial dependence plot for ADHD PRS
term = 0

XX = gam4_cimt.generate_X_grid(term=term)

pdep, confi = gam4_cimt.partial_dependence(
    term=term,
    X=XX,
    width=0.95
)

plt.figure(figsize=(10, 6))

plt.plot(
    XX[:, term],
    pdep,
    color='red',
    label='ADHD PRS smooth'
)

plt.fill_between(
    XX[:, term],
    confi[:, 0],
    confi[:, 1],
    alpha=0.3,
    color='blue',
    label='95% CI'
)

plt.axhline(
    0,
    color='black',
    linestyle='--',
    linewidth=1
)

plt.title("GAM 4: Partial Effect of ADHD PRS on CIMT")
plt.xlabel("Standardised ADHD PRS")
plt.ylabel("Partial effect on log meanmax CIMT")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# GAM 4 Linear PRS:
# CIMT ~ linear ADHD PRS + smooth age + sex + smooth BMI + smooth Townsend
#        + smoking + alcohol + smooth genetic PCs + chip

from pygam import LinearGAM, s, f, l
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Select columns
cols = [
    'prs_adhd_std',
    'age_0_0',
    'sex',
    'n_21001_0_0',      # BMI
    'townsend',
    'cursmoke_0',
    'curr_alcohol_0',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8',
    'chip',
    'nlog_meanmax_imt_mm_2'
]

df_gam4_linear = df_basic[cols].copy()

# Replace inf values with NaN
df_gam4_linear = df_gam4_linear.replace([np.inf, -np.inf], np.nan)

# Drop missing values
df_gam4_linear = df_gam4_linear.dropna().copy()

# Convert categorical variables to numeric codes
df_gam4_linear['sex'] = df_gam4_linear['sex'].astype('category').cat.codes
df_gam4_linear['cursmoke_0'] = df_gam4_linear['cursmoke_0'].astype('category').cat.codes
df_gam4_linear['curr_alcohol_0'] = df_gam4_linear['curr_alcohol_0'].astype('category').cat.codes
df_gam4_linear['chip'] = df_gam4_linear['chip'].astype('category').cat.codes

# Prepare X and y
X_gam4_linear = df_gam4_linear[
    [
        'prs_adhd_std',
        'age_0_0',
        'sex',
        'n_21001_0_0',
        'townsend',
        'cursmoke_0',
        'curr_alcohol_0',
        'PGC1', 'PGC2', 'PGC3', 'PGC4',
        'PGC5', 'PGC6', 'PGC7', 'PGC8',
        'chip'
    ]
].values

y_gam4_linear = df_gam4_linear['nlog_meanmax_imt_mm_2'].values

# Check missingness and shape
print("NaN in X:", np.isnan(X_gam4_linear).sum())
print("Inf in X:", np.isinf(X_gam4_linear).sum())
print("NaN in y:", np.isnan(y_gam4_linear).sum())
print("Inf in y:", np.isinf(y_gam4_linear).sum())
print("Shape:", X_gam4_linear.shape, y_gam4_linear.shape)

# Fit GAM
# Terms:
# l(0) = ADHD PRS as linear term
# s(1) = age
# f(2) = sex
# s(3) = BMI
# s(4) = Townsend
# f(5) = current smoking
# f(6) = current alcohol
# s(7) to s(14) = genetic PCs
# f(15) = chip

gam4_linear_cimt = LinearGAM(
    l(0) +
    s(1) +
    f(2) +
    s(3) +
    s(4) +
    f(5) +
    f(6) +
    s(7) + s(8) + s(9) + s(10) +
    s(11) + s(12) + s(13) + s(14) +
    f(15)
).fit(X_gam4_linear, y_gam4_linear)

# Print summary
gam4_linear_cimt.summary()

# Partial dependence plot for linear ADHD PRS term
term = 0

XX = gam4_linear_cimt.generate_X_grid(term=term)

pdep, confi = gam4_linear_cimt.partial_dependence(
    term=term,
    X=XX,
    width=0.95
)

plt.figure(figsize=(10, 6))

plt.plot(
    XX[:, term],
    pdep,
    color='red',
    label='Linear ADHD PRS effect'
)

plt.fill_between(
    XX[:, term],
    confi[:, 0],
    confi[:, 1],
    alpha=0.3,
    color='blue',
    label='95% CI'
)

plt.axhline(
    0,
    color='black',
    linestyle='--',
    linewidth=1
)

plt.title("GAM 4 Linear PRS: Partial Effect of ADHD PRS on CIMT")
plt.xlabel("Standardised ADHD PRS")
plt.ylabel("Partial effect on log meanmax CIMT")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# COHORT DESCRIPTION TABLE
# ============================================================

# Create sex variable exactly as in your existing analysis
# sex_plink: 1 = male, 2 = female
df['sex'] = df['sex_plink'].map({1: 0, 2: 1})


# ------------------------------------------------------------
# 1. Check likely column names for variables not in this script
# ------------------------------------------------------------

print("Possible WHR / waist / hip columns:")
print([
    col for col in df.columns
    if any(x in col.lower() for x in ['whr', 'waist', 'hip'])
])

print("\nPossible HbA1c columns:")
print([
    col for col in df.columns
    if any(x in col.lower() for x in ['hba1c', 'hba', 'glycat'])
])

print("\nPossible cIMT columns:")
print([
    col for col in df.columns
    if any(x in col.lower() for x in ['cimt', 'imt', 'carotid'])
])


# ------------------------------------------------------------
# 2. Define variables whose names are confirmed
# ------------------------------------------------------------

continuous_vars = {
    'Age, years': 'age_0_0',
    'BMI, kg/m²': 'n_21001_0_0',
    'SBP, mmHg': 'adj_sbp_0',
    'DBP, mmHg': 'adj_dbp_0',
    'Townsend deprivation index': 'townsend'
}


# ------------------------------------------------------------
# 3. Continuous-variable summary
# ------------------------------------------------------------

continuous_rows = []

for label, col in continuous_vars.items():

    x = pd.to_numeric(df[col], errors='coerce').dropna()

    continuous_rows.append({
        'Characteristic': label,
        'N': len(x),
        'Mean': x.mean(),
        'SD': x.std(),
        'Mean ± SD': f"{x.mean():.2f} ± {x.std():.2f}"
    })

continuous_table = pd.DataFrame(continuous_rows)

print("\n==============================================")
print("CONTINUOUS CHARACTERISTICS")
print("==============================================")
print(continuous_table.to_string(index=False))


# ------------------------------------------------------------
# 4. Sex
# ------------------------------------------------------------

sex_valid = df['sex'].dropna()

n_total_sex = len(sex_valid)

n_male = (sex_valid == 0).sum()
n_female = (sex_valid == 1).sum()

male_pct = (n_male / n_total_sex) * 100
female_pct = (n_female / n_total_sex) * 100

print("\n==============================================")
print("SEX")
print("==============================================")
print(f"N with sex data: {n_total_sex:,}")
print(f"Male:   {n_male:,} ({male_pct:.2f}%)")
print(f"Female: {n_female:,} ({female_pct:.2f}%)")


# ------------------------------------------------------------
# 5. Current smoking
# ------------------------------------------------------------

smoking = df['cursmoke_0'].dropna()

print("\n==============================================")
print("CURRENT SMOKING")
print("==============================================")
print(smoking.value_counts(dropna=False))
print("\nPercentages:")
print((smoking.value_counts(normalize=True) * 100).round(2))


# ------------------------------------------------------------
# 6. Current alcohol use
# ------------------------------------------------------------

alcohol = df['curr_alcohol_0'].dropna()

print("\n==============================================")
print("CURRENT ALCOHOL USE")
print("==============================================")
print(alcohol.value_counts(dropna=False))
print("\nPercentages:")
print((alcohol.value_counts(normalize=True) * 100).round(2))


# ------------------------------------------------------------
# 7. Overall source dataset size
# ------------------------------------------------------------

print("\n==============================================")
print("OVERALL DATASET")
print("==============================================")
print(f"Total rows in source dataset: {len(df):,}")
print(f"Participants with PRS-ADHD: {df['prs_adhd_std'].notna().sum():,}")